# Exercise - Customer Support Supervisor

In this exercise, you will implement a supervisor architecture using LangGraph where a supervisor agent coordinates multiple specialized support agents through tool calls, practicing the handoffs pattern you learned in the demo.

**Challenge**

You're building a customer support system with a supervisor agent that routes customer inquiries to specialized support agents:
- Supervisor Agent: Analyzes customer requests and decides which specialist to call
- Technical Support Agent: Handles technical issues and product problems
- Billing Support Agent: Manages payment, subscription, and billing questions
- General Support Agent: Handles general inquiries and account management
- Escalation Agent: Manages complex cases that need human intervention

Your solution should demonstrate:
- Supervisor agent making intelligent routing decisions
- Tool-calling pattern for agent communication
- Proper state management between agents
- Clean handoff mechanisms
- Scalable multi-agent architecture


## 0. Set up the Vocareum model

Use the project Python environment with LangChain 1.x, LangGraph, LangChain OpenAI, and python-dotenv installed. Set `VOCAREUM_API_KEY` in the environment or the project `.env` before running the cells in order.

The specialists and supervisor use [`create_agent`](https://docs.langchain.com/oss/python/migrate/langchain-v1#migrate-to-create_agent) to build their LangGraph workflows.

In [65]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt.tool_node import ToolRuntime
from langgraph.types import Command

In [66]:
load_dotenv(find_dotenv(usecwd=True))
vocareum_key = os.getenv("VOCAREUM_API_KEY")
if not vocareum_key:
    raise RuntimeError("Set VOCAREUM_API_KEY in the environment or the project .env file.")

In [67]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=vocareum_key,
    timeout=25,
    max_retries=1,
)

## 1. Create the specialized agents

Each specialist uses one tool backed by synthetic support data. The billing tool reads an example ledger and the escalation tool prepares a simulated human review. These tools have no external side effects.

In [68]:
@tool
def lookup_upload_guidance() -> str:
    """Retrieve the troubleshooting checklist for crashing uploads."""
    return (
        "Update the app and try uploading one smaller supported file. "
        "If it still crashes, record the app version, file type, and error code."
    )


@tool
def lookup_billing_activity(customer_id: str) -> str:
    """Read the sample payment records for the supplied demo customer."""
    if customer_id != "demo-001":
        return "No sample billing records match this customer."
    return "Monthly subscription: one $19 payment posted and one $19 authorization pending."


@tool
def get_password_reset_steps() -> str:
    """Retrieve the documented password reset steps."""
    return (
        "Select Forgot password on the sign-in page and follow the link sent to "
        "your registered email. Check the spam folder if the reset email is missing."
    )


@tool
def prepare_human_review() -> str:
    """Identify the simulated human review queue and the information to request."""
    return "Human review queue: support. Request the affected feature and prior troubleshooting steps."

In [69]:
technical_agent = create_agent(
    llm,
    [lookup_upload_guidance],
    name="technical_support",
    system_prompt=SystemMessage(
        content="Use lookup_upload_guidance first. Give the customer concise next steps based on its result."
    ),
)

In [70]:
billing_agent = create_agent(
    llm,
    [lookup_billing_activity],
    name="billing_support",
    system_prompt=SystemMessage(
        content=(
            "Use lookup_billing_activity with the customer ID in the supplied ticket context. "
            "Explain posted payments and pending authorizations using only the tool result. "
            "Give a next step and make no promises about payment changes or refunds."
        )
    ),
)

In [71]:
general_agent = create_agent(
    llm,
    [get_password_reset_steps],
    name="general_support",
    system_prompt=SystemMessage(
        content="Use get_password_reset_steps first. Explain the account recovery steps without requesting a password."
    ),
)

In [72]:
escalation_agent = create_agent(
    llm,
    [prepare_human_review],
    name="escalation_support",
    system_prompt=SystemMessage(
        content=(
            "Use prepare_human_review first. Summarize the issue, tell the customer it needs human review, "
            "and request only the missing details needed for that review. The ticket remains open."
        )
    ),
)

## 2. Create the supervisor tools

The supervisor passes ticket context to the selected specialist. LangGraph injects ticket and customer IDs from state, so the model only supplies the request summary. The routing tool returns a response and a state update in the same handoff.

In [73]:
class SupportState(AgentState):
    ticket_id: str
    customer_id: str
    assigned_agent: str
    resolution_status: str
    escalation_flag: bool


def handoff(agent, agent_name: str, issue: str, runtime: ToolRuntime) -> Command:
    ticket_id = runtime.state["ticket_id"]
    customer_id = runtime.state["customer_id"]
    result = agent.invoke(
        {"messages": [HumanMessage(
            content=f"Ticket {ticket_id} for customer {customer_id}.\nRequest: {issue}"
        )]},
        config={"recursion_limit": 12},
    )

    needs_human = agent_name == "escalation"
    state_update = {
        "ticket_id": ticket_id,
        "customer_id": customer_id,
        "assigned_agent": agent_name,
        "resolution_status": "pending_human_review" if needs_human else "answered",
        "escalation_flag": needs_human,
    }
    payload = {
        **state_update,
        "response": result["messages"][-1].text,
        "specialist_tools": [
            message.name for message in result["messages"] if isinstance(message, ToolMessage)
        ],
    }
    return Command(
        update={
            **state_update,
            "messages": [ToolMessage(
                content=json.dumps(payload),
                tool_call_id=runtime.tool_call_id,
            )],
        }
    )


@tool(return_direct=True)
def route_to_technical(issue_description: str, runtime: ToolRuntime) -> Command:
    """Delegate an app crash, upload failure, or other product malfunction."""
    return handoff(technical_agent, "technical", issue_description, runtime)

In [74]:
@tool(return_direct=True)
def route_to_billing(billing_question: str, runtime: ToolRuntime) -> Command:
    """Delegate questions about payments, subscriptions, or duplicate charges."""
    return handoff(billing_agent, "billing", billing_question, runtime)

In [75]:
@tool(return_direct=True)
def route_to_general(query: str, runtime: ToolRuntime) -> Command:
    """Delegate routine account questions such as resetting a password."""
    return handoff(general_agent, "general", query, runtime)


@tool(return_direct=True)
def route_to_escalation(details: str, runtime: ToolRuntime) -> Command:
    """Send a cross-domain, unclear, or repeatedly unresolved request for human review."""
    return handoff(escalation_agent, "escalation", details, runtime)

## 3. Supervisor Agent

The supervisor calls one routing tool per ticket. A routing tool returns the specialist's answer or escalation summary directly, along with the updated state.

In [76]:
@wrap_model_call
def require_single_handoff(request, handler):
    return handler(request.override(
        tool_choice="required",
        model_settings={**request.model_settings, "parallel_tool_calls": False},
    ))


routing_tools = [
    route_to_technical,
    route_to_billing,
    route_to_general,
    route_to_escalation,
]
supervisor_agent = create_agent(
    name="support_supervisor",
    system_prompt=SystemMessage(
        content=(
            "You coordinate customer support. Choose exactly one routing tool and give it "
            "a faithful summary of the customer's request. Use technical for an app crash "
            "or upload failure, billing for suspected duplicate charges, and general for "
            "password reset or routine account help. Escalate cross-domain requests, requests "
            "with too little information to assign a domain, and vague reports that all "
            "troubleshooting has failed. Do not answer the customer's question yourself."
        )
    ),
    model=llm,
    middleware=[require_single_handoff],
    tools=routing_tools,
    state_schema=SupportState,
)

## 4. Test the routing scenarios

Run the four sample tickets from the exercise, followed by an ambiguous request and a mixed billing and technical request. Each invocation starts with fresh state.

In [77]:
scenarios = [
    ("TKT-001", "My app keeps crashing when I try to upload files", "technical", "lookup_upload_guidance"),
    ("TKT-002", "Why was I charged twice this month?", "billing", "lookup_billing_activity"),
    ("TKT-003", "How do I change my account password?", "general", "get_password_reset_steps"),
    ("TKT-004", "I've tried everything but nothing works", "escalation", "prepare_human_review"),
    ("TKT-005", "Can someone help me?", "escalation", "prepare_human_review"),
    ("TKT-006", "The app crashes during uploads and I was charged twice.", "escalation", "prepare_human_review"),
]

In [78]:
def run_ticket(ticket_id: str, customer_id: str, message: str) -> dict:
    return supervisor_agent.invoke(
        {
            "messages": [HumanMessage(content=message)],
            "ticket_id": ticket_id,
            "customer_id": customer_id,
            "assigned_agent": "unassigned",
            "resolution_status": "open",
            "escalation_flag": False,
        },
        config={"recursion_limit": 12},
    )


customer_id = "demo-001"
results = {
    ticket_id: run_ticket(ticket_id, customer_id, message)
    for ticket_id, message, _, _ in scenarios
}

## 5. Verify state and handoffs

Check the routing tool, specialist tool, ticket identity, and final status for each scenario. `answered` means the specialist provided guidance. `pending_human_review` keeps the ticket open for follow-up.

In [79]:
for ticket_id, _, expected_agent, expected_tool in scenarios:
    result = results[ticket_id]
    handoffs = [message for message in result["messages"] if isinstance(message, ToolMessage)]
    assert len(handoffs) == 1, f"{ticket_id}: expected exactly one handoff"
    route = handoffs[0]
    payload = json.loads(route.content)
    needs_human = expected_agent == "escalation"
    expected_state = {
        "ticket_id": ticket_id,
        "customer_id": customer_id,
        "assigned_agent": expected_agent,
        "resolution_status": "pending_human_review" if needs_human else "answered",
        "escalation_flag": needs_human,
    }

    assert route.name == f"route_to_{expected_agent}", f"{ticket_id}: unexpected route {route.name}"
    assert expected_tool in payload["specialist_tools"], f"{ticket_id}: specialist skipped its tool"
    assert {key: result[key] for key in expected_state} == expected_state, ticket_id
    assert {key: payload[key] for key in expected_state} == expected_state, ticket_id
    assert payload["response"].strip(), f"{ticket_id}: empty response"

    print(f"{ticket_id} | {route.name} → {expected_tool} | {result['resolution_status']}")
    print(f"  {payload['response']}")

print(f"Passed {len(scenarios)} scenarios.")

TKT-001 | route_to_technical → lookup_upload_guidance | answered
  Here are the next steps to resolve the issue with the app crashing during file uploads:

1. **Update the App**: Ensure that you have the latest version of the app installed.
2. **Try Uploading a Smaller File**: Attempt to upload a smaller file that is supported by the app.
3. **If It Still Crashes**: If the app continues to crash, please record the following details:
   - App version
   - File type being uploaded
   - Any error code displayed

Let me know if you need further assistance!
TKT-002 | route_to_billing → lookup_billing_activity | answered
  For customer demo-001, there is one posted payment of $19 for the monthly subscription, and there is also a pending authorization for another $19. 

**Posted Payments:** This indicates that the payment of $19 has been successfully processed and is reflected in the account.

**Pending Authorizations:** This means that a charge of $19 is currently being held but has not yet 